# ⚙️ Notebook 02 — Advanced Feature Engineering (V2)
## ParkSense AI | Gridlock Round 2

**Input:** `data/processed/clean_violations.parquet`  
**Output:** `data/processed/featured_violations.parquet`

This notebook generates mathematically robust features, including:
1. Geospatial & Temporal identifiers.
2. The absolute-scaled **Parking-Induced Congestion Impact (PICI)** Score.
3. Strict Cumulative Repeat Offender tracking (preventing data leakage).
4. Indian Public Holiday context integration.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

print('✅ Imports done.')

✅ Imports done.


In [8]:
import json
CLEAN_PATH = '../data/processed/clean_violations.parquet'
df = pd.read_parquet(CLEAN_PATH)

def safe_load(x):
    if isinstance(x, list): return x
    try: return json.loads(x)
    except:
        import ast
        try: return ast.literal_eval(x)
        except: return []

df['violation_list'] = df['violation_list'].apply(safe_load)
if 'offence_code_list' in df.columns:
    df['offence_code_list'] = df['offence_code_list'].apply(safe_load)

print(f'Loaded: {df.shape[0]:,} rows')

Loaded: 115,353 rows


## 1. Temporal & Holiday Fixes

In [9]:
# Ensure datetime
if 'created_datetime' not in df.columns or not pd.api.types.is_datetime64_any_dtype(df['created_datetime']):
    df['created_datetime'] = pd.to_datetime(df['created_datetime'], utc=True).dt.tz_convert('Asia/Kolkata')

df['hour'] = df['created_datetime'].dt.hour
df['day_of_week'] = df['created_datetime'].dt.dayofweek
df['date'] = df['created_datetime'].dt.date
df['month'] = df['created_datetime'].dt.month
df['is_weekend'] = df['day_of_week'].apply(lambda d: 1 if d >= 5 else 0)

# Inject Indian Public Holidays (Nov 2023 - May 2024)
HOLIDAYS = [
    '2023-11-12', # Diwali
    '2023-11-13', # Diwali Holiday
    '2023-12-25', # Christmas
    '2024-01-01', # New Year
    '2024-01-15', # Makar Sankranti
    '2024-01-26', # Republic Day
    '2024-03-08', # Maha Shivaratri
    '2024-03-25', # Holi
    '2024-04-09', # Ugadi
    '2024-04-11', # Eid
    '2024-05-01'  # May Day
]
holidays_dt = pd.to_datetime(HOLIDAYS).date
df['is_holiday'] = df['date'].isin(holidays_dt).astype(int)

# Peak hour ONLY applies on Weekdays AND Non-Holidays
df['is_peak_hour'] = df.apply(
    lambda r: 1 if (r['is_weekend'] == 0 and r['is_holiday'] == 0) and ((8 <= r['hour'] <= 11) or (17 <= r['hour'] <= 21)) else 0,
    axis=1
)

df['is_business_hours'] = df.apply(
    lambda r: 1 if (r['is_weekend'] == 0 and r['is_holiday'] == 0) and (9 <= r['hour'] < 18) else 0,
    axis=1
)

print(f"Total violations on Public Holidays: {df['is_holiday'].sum():,}")

Total violations on Public Holidays: 6,765


## 2. PICI Score with Absolute Scaling

In [10]:
# Vehicle Size
VEHICLE_SIZE_FACTOR = {
    'TANKER': 2.0, 'BUS': 2.0, 'TRUCK': 2.0, 'LGV': 2.0,
    'CAR': 1.5, 'MAXI-CAB': 1.5, 'VAN': 1.5,
    'PASSENGER AUTO': 1.2, 'GOODS AUTO': 1.2,
    'SCOOTER': 1.0, 'MOTOR CYCLE': 1.0, 'MOPED': 1.0, 'OTHERS': 1.0,
}
VEHICLE_CATEGORY = {
    'TANKER': 'HEAVY', 'BUS': 'HEAVY', 'TRUCK': 'HEAVY', 'LGV': 'HEAVY',
    'CAR': 'MEDIUM', 'MAXI-CAB': 'MEDIUM', 'VAN': 'MEDIUM',
    'PASSENGER AUTO': 'LIGHT', 'GOODS AUTO': 'LIGHT',
    'SCOOTER': 'TWO_WHEELER', 'MOTOR CYCLE': 'TWO_WHEELER', 'MOPED': 'TWO_WHEELER',
    'OTHERS': 'UNKNOWN'
}
df['vehicle_size_factor'] = df['final_vehicle_type'].map(VEHICLE_SIZE_FACTOR).fillna(1.0)
df['vehicle_category'] = df['final_vehicle_type'].map(VEHICLE_CATEGORY).fillna('UNKNOWN')

# Severity Weight
SEVERITY_WEIGHTS = {
    'DOUBLE PARKING': 10, 'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE': 9,
    'PARKING IN A MAIN ROAD': 8, 'PARKING NEAR ROAD CROSSING': 7,
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC': 6, 'WRONG PARKING': 4,
    'NO PARKING': 3, 'DEFECTIVE NUMBER PLATE': 0,
}
df['severity_score'] = df['violation_list'].apply(lambda lst: sum(SEVERITY_WEIGHTS.get(v, 2) for v in lst if isinstance(lst, list)))

df['parking_violation_count'] = df['violation_list'].apply(lambda lst: sum(1 for v in lst if v != 'DEFECTIVE NUMBER PLATE' and isinstance(lst, list)))
df['multi_vio_factor'] = df['parking_violation_count'].apply(lambda n: min(1.0 + 0.1 * max(0, n - 1), 1.5))

if 'has_junction' not in df.columns:
    df['has_junction'] = df['junction_name'].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == 'No Junction' else 1)
df['junction_multiplier'] = df['has_junction'].map({1: 2.0, 0: 1.0})
df['peak_hour_multiplier'] = df['is_peak_hour'].map({1: 1.5, 0: 1.0})

# Raw PICI
df['pici_raw'] = df['severity_score'] * df['vehicle_size_factor'] * df['junction_multiplier'] * df['peak_hour_multiplier'] * df['multi_vio_factor']

# Absolute Maximum Scaling
pici_max = df['pici_raw'].max()
df['pici_score'] = ((df['pici_raw'] / pici_max) * 10).round(3)

print('PICI Score statistics (Absolute Scaling):')
print(df['pici_score'].describe().round(3))

PICI Score statistics (Absolute Scaling):
count    115353.000
mean          0.342
std           0.283
min           0.111
25%           0.178
50%           0.296
75%           0.444
max          10.000
Name: pici_score, dtype: float64


## 3. Cumulative Repeat Offenders (No Data Leakage)

In [11]:
if 'final_vehicle_number' not in df.columns:
    df['final_vehicle_number'] = df['updated_vehicle_number'].fillna(df['vehicle_number'])

# Sort chronologically and use cumcount()
df = df.sort_values(['final_vehicle_number', 'created_datetime']).reset_index(drop=True)
df['vehicle_prior_violations'] = df.groupby('final_vehicle_number').cumcount()

df['is_repeat_offender'] = (df['vehicle_prior_violations'] >= 3).astype(int)
df['is_chronic_offender'] = (df['vehicle_prior_violations'] >= 10).astype(int)

print(f"Repeat offense events: {df['is_repeat_offender'].sum():,}")

Repeat offense events: 3,421


In [12]:

# Grid & Other features
df['lat_grid'] = df['latitude'].round(3)
df['lng_grid'] = df['longitude'].round(3)
df['grid_cell'] = df['lat_grid'].astype(str) + '_' + df['lng_grid'].astype(str)

def extract_pincode(x):
    m = re.search(r'Pin-(\d{6})', str(x))
    return m.group(1) if m else None
df['pincode'] = df['location'].apply(extract_pincode)

OUT_PATH = '../data/processed/featured_violations.parquet'
df.to_parquet(OUT_PATH, index=False)
print(f'✅ Featured dataset saved: {OUT_PATH}')


✅ Featured dataset saved: ../data/processed/featured_violations.parquet
